[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%203/ex1_text_classification.ipynb)

# ISBA 2411 - In-Class Exercise 1: The feature face-off

**Supervised Text Classification & Representation - Week 3, Lecture 5**

**Goal (~15 min, pairs).** Build a baseline text classifier, change exactly ONE thing about how the text is turned into features, and measure whether your change actually helped on data the model has never seen.

This is the same shape as **Milestone 2**: a baseline, one deliberate improvement, and a number that says whether it worked.

**Setup.** Pure Python, no API key. Run the warm-up cell once so the runtime is ready, then work top to bottom. One person drives, one person reads the numbers out loud and writes down the answers.

In [ ]:
# Warm up the runtime and confirm the packages are present.
!pip -q install datasets scikit-learn 2>/dev/null
import sklearn, datasets
print('scikit-learn', sklearn.__version__)
print('datasets', datasets.__version__)

---
## 1. Load a small labeled review set

We use **rotten_tomatoes**: short movie-review sentences, each labeled `0` (negative) or `1` (positive). It is small (about 8,500 training reviews, 1,000 test), already split into train and test, and downloads in seconds.

> If the room wifi blocks the download, skip to the **Fallback** section at the bottom, run that one cell, then come back here.

In [ ]:
from datasets import load_dataset

ds = load_dataset('rotten_tomatoes')
train_text, train_label = ds['train']['text'], ds['train']['label']
test_text,  test_label  = ds['test']['text'],  ds['test']['label']

labels = {0: 'negative', 1: 'positive'}
print('train reviews:', len(train_text), '  test reviews:', len(test_text))
print('\nexample:', labels[train_label[0]], '->', train_text[0])

---
## 2. Build the baseline

The recipe is the one from lecture: turn each review into a **TF-IDF** vector, then train a **logistic regression** classifier.

The baseline keeps every knob plain on purpose: unigrams only, no `min_df`, no stopword list. Those are exactly the knobs you will change in Step 3.

Notice we print **two** numbers. The model has already seen the training reviews, so its training accuracy is always flattering. Only the **test** number tells you anything real.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

vec = TfidfVectorizer()            # plain baseline: the knobs to change all live here
Xtr = vec.fit_transform(train_text)
Xte = vec.transform(test_text)

clf = LogisticRegression(max_iter=1000)
clf.fit(Xtr, train_label)

train_acc = accuracy_score(train_label, clf.predict(Xtr))
baseline_acc = accuracy_score(test_label, clf.predict(Xte))

print(f'train accuracy: {train_acc:.4f}   (the model already saw this data)')
print(f'TEST  accuracy: {baseline_acc:.4f}   <-- write this number down')
print(f'\nfeatures (vocabulary size): {len(vec.vocabulary_):,}')

**Pause and look at the gap.** Training accuracy sits well above test accuracy. A model that scored near-perfect on data it trained on can score far lower on data it has never seen. The lesson for your project: report the **test** number, never the training one.

Baseline test accuracy is your reference point. Everything in the next step is measured against it.

---
## 3. Make ONE deliberate change

Pick **exactly one** knob and change it. Keep everything else identical, so any movement in the score is caused by your one change and nothing else.

Three options, each a different idea about representation:

| Change | What it does | What you might expect |
|---|---|---|
| `ngram_range=(1, 2)` | adds **bigrams**, so word pairs become features | catches phrases like *not good* that unigrams miss |
| `min_df=2` | drops terms that appear in fewer than 2 reviews | removes rare noise, shrinks the feature space |
| `stop_words='english'` | removes very common words (*the*, *a*, *is*) | fewer features, but you might delete real signal |

Uncomment **one** line below, leave the other two commented, then run.

In [ ]:
# Change exactly ONE thing. Uncomment ONE line; leave the others commented.

vec2 = TfidfVectorizer(ngram_range=(1, 2))      # add bigrams
# vec2 = TfidfVectorizer(min_df=2)              # drop very rare terms
# vec2 = TfidfVectorizer(stop_words='english')  # remove common words

Xtr2 = vec2.fit_transform(train_text)
Xte2 = vec2.transform(test_text)
clf2 = LogisticRegression(max_iter=1000).fit(Xtr2, train_label)
new_acc = accuracy_score(test_label, clf2.predict(Xte2))

print(f'baseline TEST accuracy : {baseline_acc:.4f}')
print(f'your change TEST acc.  : {new_acc:.4f}')
print(f'change                 : {new_acc - baseline_acc:+.4f}')
print(f'features now            : {len(vec2.vocabulary_):,}  (baseline was {len(vec.vocabulary_):,})')

Did it help, hurt, or do almost nothing? Not every reasonable change improves the test score. That surprise is the point. If you have time, reset the cell and try a different option to compare.

---
## 4. Find a review the model still gets wrong

A single accuracy number hides the individual mistakes. Pull out a few reviews your improved model still gets wrong, and read them. Some are genuinely hard; some are plain errors a person would not make.

In [ ]:
pred2 = clf2.predict(Xte2)
wrong = [i for i in range(len(test_label)) if pred2[i] != test_label[i]]
print(f'{len(wrong)} of {len(test_label)} test reviews are still wrong\n')

for i in wrong[:6]:
    print(f"[true {labels[test_label[i]]:>8} | pred {labels[pred2[i]]:>8}]  {test_text[i][:130]}")

In [ ]:
# Now try your own. Write a couple of reviews and see how the model labels them.
my_reviews = [
    'this is not a good film',
    'a sharp, funny movie that earns its ending',
]
for s, p in zip(my_reviews, clf2.predict(vec2.transform(my_reviews))):
    print(f'{labels[p]:>8}  <-  {s}')

---
## Bring back an answer to

Be ready to report these out:

1. **Which single change moved accuracy the most, and why?**
2. **Did your "improvement" help on the test set, or only on training?**
3. **What kind of review fools the model every time?** Share one example you found.

**Connection to your final project.** What you just did is Milestone 2 in miniature: a baseline, one deliberate change, and a metric measured on held-out data. Reuse this notebook as the starting scaffold for your own dataset.

---
## Fallback (only if the dataset will not download)

If `load_dataset` is blocked, upload a `reviews.csv` with two columns, `text` and `label` (0 or 1), to the Colab file panel, then run the cell below instead of Section 1. Everything after it stays the same.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('reviews.csv')
train_text, test_text, train_label, test_label = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)
labels = {0: 'negative', 1: 'positive'}
print('train reviews:', len(train_text), '  test reviews:', len(test_text))

---
### Reading connection

| Idea tonight | Where to read more |
|---|---|
| Text classification, sentiment, evaluation | J&M ch. 4 |
| Bag of words, TF-IDF, the term-document matrix | J&M ch. 4; Tunstall ch. 2 |
| Classification with the Hugging Face toolkit | HOLLM ch. 4 |

*ISBA 2411 - Natural Language Processing & AI - Summer 2026*